In [0]:
%sql
  CREATE WIDGET TEXT catalog DEFAULT 'bootcamp';
  USE CATALOG IDENTIFIER(:catalog);

# Phase 3 · Gold dimensions
Builds the star schema from silver.
- **Dimensions** use surrogate keys (`BIGINT GENERATED BY DEFAULT AS IDENTITY`) and PRIMARY KEY constraints. They are loaded with **MERGE** on the business key (SCD Type 1): new rows are inserted, changed rows are overwritten, so keys stay stable and re-runs change nothing. `dim_date` (key = yyyyMMdd) is generated with `sequence` + `explode` and loaded with `INSERT OVERWRITE`.
- **Unknown member −1** only where needed: `dim_customer` (guest orders) and `dim_product` (PX000). Not in `dim_store` or `dim_date`, because silver rejects rows that would need them. `P9999` is excluded from `dim_product`.
- **fact_sales** (grain = one order line) gets its surrogate keys through lookups on the dimensions (`coalesce(..., -1)` for customer and product), has FOREIGN KEY constraints, `CLUSTER BY (date_key)`, and enforced CHECK constraints (`quantity > 0`). Cost and margin are calculated with the product's unit cost.

In [0]:
CREATE TABLE IF NOT EXISTS gold_efkleia_urbanmart.dim_date (
  date_key        INT     NOT NULL COMMENT 'Surrogate key in yyyyMMdd format.',
  full_date       DATE    NOT NULL COMMENT 'Calendar date.',
  day_of_month    INT     NOT NULL COMMENT 'Day number in month (1-31).',
  day_name        STRING  NOT NULL COMMENT 'Monday ... Sunday.',
  iso_day_of_week INT     NOT NULL COMMENT '1 = Monday ... 7 = Sunday.',
  is_weekend      BOOLEAN NOT NULL COMMENT 'TRUE for Saturday and Sunday.',
  week_of_year    INT     NOT NULL COMMENT 'ISO week number.',
  month_num       INT     NOT NULL COMMENT 'Month number (1-12).',
  month_name      STRING  NOT NULL COMMENT 'January ... December.',
  quarter_num     INT     NOT NULL COMMENT 'Calendar quarter (1-4).',
  year_num        INT     NOT NULL COMMENT 'Calendar year.',
  year_month      STRING  NOT NULL COMMENT 'yyyy-MM, for monthly reporting.',
  CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
)
COMMENT 'Gold: calendar dimension 2024-01-01 to 2025-12-31.';

INSERT OVERWRITE gold_efkleia_urbanmart.dim_date
  (date_key, full_date, day_of_month, day_name, iso_day_of_week, is_weekend,
   week_of_year, month_num, month_name, quarter_num, year_num, year_month)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT),
  d,
  dayofmonth(d),
  date_format(d, 'EEEE'),
  weekday(d) + 1,
  weekday(d) >= 5,
  weekofyear(d),
  month(d),
  date_format(d, 'MMMM'),
  quarter(d),
  year(d),
  date_format(d, 'yyyy-MM')
FROM (SELECT explode(sequence(DATE'2024-01-01', DATE'2025-12-31', INTERVAL 1 DAY)) AS d);

num_affected_rows,num_inserted_rows
731,731


In [0]:
CREATE TABLE IF NOT EXISTS gold_efkleia_urbanmart.dim_customer (
  customer_key   BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 1 INCREMENT BY 1)
                            COMMENT 'Surrogate key; -1 = Guest / Unknown.',
  customer_id    STRING    NOT NULL COMMENT 'Business key from source.',
  first_name     STRING             COMMENT 'First name.',
  last_name      STRING             COMMENT 'Last name.',
  full_name      STRING             COMMENT 'First + last name, for reporting.',
  email          STRING             COMMENT 'Email; NULL when unknown.',
  gender         STRING             COMMENT 'F / M.',
  birth_date     DATE               COMMENT 'Birth date.',
  city           STRING             COMMENT 'City.',
  region         STRING             COMMENT 'Region.',
  segment        STRING             COMMENT 'Retail / Loyalty Silver / Loyalty Gold / Guest.',
  signup_date    DATE               COMMENT 'Signup date.',
  _attr_hash     STRING    NOT NULL COMMENT 'SHA-256 of attributes; drives SCD1 change detection.',
  _inserted_at   TIMESTAMP NOT NULL COMMENT 'Row insert timestamp.',
  _updated_at    TIMESTAMP          COMMENT 'Last SCD1 overwrite timestamp.',
  CONSTRAINT pk_dim_customer PRIMARY KEY (customer_key)
)
COMMENT 'Gold: customer dimension, SCD Type 1, with unknown member -1 for guest orders.';

In [0]:
MERGE INTO gold_efkleia_urbanmart.dim_customer AS t
USING (SELECT CAST(-1 AS BIGINT) AS customer_key) AS s
ON t.customer_key = s.customer_key
WHEN NOT MATCHED THEN INSERT
  (customer_key, customer_id, first_name, last_name, full_name, email, gender, birth_date,
   city, region, segment, signup_date, _attr_hash, _inserted_at)
VALUES
  (-1, 'UNKNOWN', NULL, NULL, 'Guest / Unknown', NULL, NULL, NULL,
   'Unknown', 'Unknown', 'Guest', NULL, sha2('unknown', 256), current_timestamp());

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
MERGE INTO gold_efkleia_urbanmart.dim_customer AS t
USING (
  SELECT
    customer_id, first_name, last_name,
    concat_ws(' ', first_name, last_name) AS full_name,
    email, gender, birth_date, city, region, segment, signup_date,
    sha2(concat_ws('||', coalesce(first_name, ''), coalesce(last_name, ''), coalesce(email, ''),
                         coalesce(gender, ''), coalesce(CAST(birth_date AS STRING), ''),
                         coalesce(city, ''), coalesce(region, ''), coalesce(segment, ''),
                         coalesce(CAST(signup_date AS STRING), '')), 256) AS attr_hash
  FROM silver_efkleia_urbanmart.customers
) AS s
ON t.customer_id = s.customer_id
WHEN MATCHED AND t._attr_hash <> s.attr_hash THEN UPDATE SET
  t.first_name  = s.first_name,
  t.last_name   = s.last_name,
  t.full_name   = s.full_name,
  t.email       = s.email,
  t.gender      = s.gender,
  t.birth_date  = s.birth_date,
  t.city        = s.city,
  t.region      = s.region,
  t.segment     = s.segment,
  t.signup_date = s.signup_date,
  t._attr_hash  = s.attr_hash,
  t._updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT
  (customer_id, first_name, last_name, full_name, email, gender, birth_date,
   city, region, segment, signup_date, _attr_hash, _inserted_at)
VALUES
  (s.customer_id, s.first_name, s.last_name, s.full_name, s.email, s.gender, s.birth_date,
   s.city, s.region, s.segment, s.signup_date, s.attr_hash, current_timestamp());

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
CREATE TABLE IF NOT EXISTS gold_efkleia_urbanmart.dim_customer_scd2 (
  customer_scd2_key BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 1 INCREMENT BY 1)
                              COMMENT 'Surrogate key, one per customer version; -1 = Guest / Unknown.',
  customer_id   STRING    NOT NULL COMMENT 'Business key.',
  full_name     STRING             COMMENT 'Customer name (not tracked).',
  email         STRING             COMMENT 'Email (not tracked).',
  gender        STRING             COMMENT 'F / M (not tracked).',
  city          STRING             COMMENT 'City for this version (SCD2 tracked).',
  region        STRING             COMMENT 'Region for this version (SCD2 tracked).',
  segment       STRING             COMMENT 'Segment for this version (SCD2 tracked).',
  valid_from    DATE      NOT NULL COMMENT 'First day this version is valid.',
  valid_to      DATE      NOT NULL COMMENT 'Last day this version is valid; 9999-12-31 when open.',
  is_current    BOOLEAN   NOT NULL COMMENT 'TRUE for the open version.',
  version_num   INT       NOT NULL COMMENT 'Version sequence per customer, starting at 1.',
  _attr_hash    STRING    NOT NULL COMMENT 'SHA-256 of all stored attributes; drives MERGE change detection.',
  _inserted_at  TIMESTAMP NOT NULL COMMENT 'Row insert timestamp.',
  _updated_at   TIMESTAMP          COMMENT 'Last update timestamp.',
  CONSTRAINT pk_dim_customer_scd2 PRIMARY KEY (customer_scd2_key)
)
COMMENT 'Gold: customer dimension, SCD Type 2 on city, region and segment (source: customer_changes).';

ALTER TABLE gold_efkleia_urbanmart.dim_customer_scd2 DROP CONSTRAINT IF EXISTS ck_scd2_dates;
ALTER TABLE gold_efkleia_urbanmart.dim_customer_scd2 ADD  CONSTRAINT ck_scd2_dates CHECK (valid_to >= valid_from);

In [0]:
MERGE INTO gold_efkleia_urbanmart.dim_customer_scd2 AS t
USING (SELECT CAST(-1 AS BIGINT) AS customer_scd2_key) AS s
ON t.customer_scd2_key = s.customer_scd2_key
WHEN NOT MATCHED THEN INSERT
  (customer_scd2_key, customer_id, full_name, email, gender, city, region, segment,
   valid_from, valid_to, is_current, version_num, _attr_hash, _inserted_at)
VALUES
  (-1, 'UNKNOWN', 'Guest / Unknown', NULL, NULL, 'Unknown', 'Unknown', 'Guest',
   DATE'1900-01-01', DATE'9999-12-31', true, 1, sha2('unknown', 256), current_timestamp());

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,0,0,1


In [0]:
MERGE INTO gold_efkleia_urbanmart.dim_customer_scd2 AS t
USING (
  WITH events AS (
    -- version 1: each customer's original values, valid from the beginning of time
    SELECT customer_id, city, region, segment, DATE'1900-01-01' AS valid_from
    FROM silver_efkleia_urbanmart.customers
    UNION ALL
    -- later versions: one per change event
    SELECT ch.customer_id, ch.new_city, ch.new_region, ch.new_segment, ch.effective_date
    FROM silver_efkleia_urbanmart.customer_changes ch
    JOIN silver_efkleia_urbanmart.customers c ON ch.customer_id = c.customer_id
  ),
  compared AS (
    SELECT *,
      sha2(concat_ws('||', coalesce(city, ''), coalesce(region, ''), coalesce(segment, '')), 256) AS tracked_hash,
      lag(sha2(concat_ws('||', coalesce(city, ''), coalesce(region, ''), coalesce(segment, '')), 256))
        OVER (PARTITION BY customer_id ORDER BY valid_from)                                     AS prev_tracked_hash
    FROM events
  ),
  versions AS (
    -- drop "changes" that change nothing (e.g. C0282)
    SELECT customer_id, city, region, segment, valid_from
    FROM compared
    WHERE prev_tracked_hash IS NULL OR prev_tracked_hash <> tracked_hash
  ),
  dated AS (
    SELECT
      v.customer_id,
      concat_ws(' ', c.first_name, c.last_name)                                         AS full_name,
      c.email,
      c.gender,
      v.city, v.region, v.segment,
      v.valid_from,
      coalesce(date_sub(lead(v.valid_from) OVER (PARTITION BY v.customer_id ORDER BY v.valid_from), 1),
               DATE'9999-12-31')                                                        AS valid_to,
      lead(v.valid_from) OVER (PARTITION BY v.customer_id ORDER BY v.valid_from) IS NULL AS is_current,
      CAST(row_number() OVER (PARTITION BY v.customer_id ORDER BY v.valid_from) AS INT)  AS version_num
    FROM versions v
    JOIN silver_efkleia_urbanmart.customers c ON v.customer_id = c.customer_id
  )
  SELECT *,
    sha2(concat_ws('||', coalesce(full_name, ''), coalesce(email, ''), coalesce(gender, ''),
                         coalesce(city, ''), coalesce(region, ''), coalesce(segment, ''),
                         CAST(valid_to AS STRING), CAST(is_current AS STRING),
                         CAST(version_num AS STRING)), 256) AS attr_hash
  FROM dated
) AS s
ON  t.customer_id = s.customer_id
AND t.valid_from  = s.valid_from
WHEN MATCHED AND t._attr_hash <> s.attr_hash THEN UPDATE SET
  t.full_name   = s.full_name,
  t.email       = s.email,
  t.gender      = s.gender,
  t.city        = s.city,
  t.region      = s.region,
  t.segment     = s.segment,
  t.valid_to    = s.valid_to,
  t.is_current  = s.is_current,
  t.version_num = s.version_num,
  t._attr_hash  = s.attr_hash,
  t._updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT
  (customer_id, full_name, email, gender, city, region, segment,
   valid_from, valid_to, is_current, version_num, _attr_hash, _inserted_at)
VALUES
  (s.customer_id, s.full_name, s.email, s.gender, s.city, s.region, s.segment,
   s.valid_from, s.valid_to, s.is_current, s.version_num, s.attr_hash, current_timestamp());

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
524,0,0,524


In [0]:
CREATE TABLE IF NOT EXISTS gold_efkleia_urbanmart.dim_product (
  product_key    BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 1 INCREMENT BY 1)
                                COMMENT 'Surrogate key; -1 = Unknown product.',
  product_id     STRING        NOT NULL COMMENT 'Business key from source.',
  product_name   STRING                 COMMENT 'Product name.',
  category       STRING                 COMMENT 'Category.',
  subcategory    STRING                 COMMENT 'Subcategory.',
  brand          STRING                 COMMENT 'Brand.',
  unit_cost      DECIMAL(10,2)          COMMENT 'Purchase cost per unit.',
  list_price     DECIMAL(10,2)          COMMENT 'Selling list price per unit.',
  is_active      BOOLEAN                COMMENT 'Product currently active.',
  _attr_hash     STRING        NOT NULL COMMENT 'SHA-256 of attributes; drives SCD1 change detection.',
  _inserted_at   TIMESTAMP     NOT NULL COMMENT 'Row insert timestamp.',
  _updated_at    TIMESTAMP              COMMENT 'Last SCD1 overwrite timestamp.',
  CONSTRAINT pk_dim_product PRIMARY KEY (product_key)
)
COMMENT 'Gold: product dimension, SCD Type 1, with unknown member -1. Source placeholder P9999 excluded.';

MERGE INTO gold_efkleia_urbanmart.dim_product AS t
USING (SELECT CAST(-1 AS BIGINT) AS product_key) AS s
ON t.product_key = s.product_key
WHEN NOT MATCHED THEN INSERT
  (product_key, product_id, product_name, category, subcategory, brand,
   unit_cost, list_price, is_active, _attr_hash, _inserted_at)
VALUES
  (-1, 'UNKNOWN', 'Unknown Product', 'Unknown', 'Unknown', 'Unknown',
   0, 0, false, sha2('unknown', 256), current_timestamp());

MERGE INTO gold_efkleia_urbanmart.dim_product AS t
USING (
  SELECT
    product_id, product_name, category, subcategory, brand, unit_cost, list_price, is_active,
    sha2(concat_ws('||', coalesce(product_name, ''), coalesce(category, ''), coalesce(subcategory, ''),
                         coalesce(brand, ''), coalesce(CAST(unit_cost AS STRING), ''),
                         coalesce(CAST(list_price AS STRING), ''), coalesce(CAST(is_active AS STRING), '')), 256) AS attr_hash
  FROM silver_efkleia_urbanmart.products
  WHERE product_id <> 'P9999'          -- source placeholder: represented by member -1
) AS s
ON t.product_id = s.product_id
WHEN MATCHED AND t._attr_hash <> s.attr_hash THEN UPDATE SET
  t.product_name = s.product_name,
  t.category     = s.category,
  t.subcategory  = s.subcategory,
  t.brand        = s.brand,
  t.unit_cost    = s.unit_cost,
  t.list_price   = s.list_price,
  t.is_active    = s.is_active,
  t._attr_hash   = s.attr_hash,
  t._updated_at  = current_timestamp()
WHEN NOT MATCHED THEN INSERT
  (product_id, product_name, category, subcategory, brand, unit_cost, list_price, is_active,
   _attr_hash, _inserted_at)
VALUES
  (s.product_id, s.product_name, s.category, s.subcategory, s.brand, s.unit_cost, s.list_price, s.is_active,
   s.attr_hash, current_timestamp());

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
CREATE TABLE IF NOT EXISTS gold_efkleia_urbanmart.dim_store (
  store_key      BIGINT GENERATED BY DEFAULT AS IDENTITY (START WITH 1 INCREMENT BY 1)
                            COMMENT 'Surrogate key.',
  store_id       STRING    NOT NULL COMMENT 'Business key from source.',
  store_name     STRING             COMMENT 'Store name.',
  store_type     STRING             COMMENT 'High Street / Mall / Outlet / Online.',
  channel        STRING             COMMENT 'Online / Physical, derived from store_type (Q5).',
  city           STRING             COMMENT 'City.',
  region         STRING             COMMENT 'Region.',
  open_date      DATE               COMMENT 'Opening date.',
  size_sqm       INT                COMMENT 'Floor area in m2; 0 for online.',
  _attr_hash     STRING    NOT NULL COMMENT 'SHA-256 of attributes; drives SCD1 change detection.',
  _inserted_at   TIMESTAMP NOT NULL COMMENT 'Row insert timestamp.',
  _updated_at    TIMESTAMP          COMMENT 'Last SCD1 overwrite timestamp.',
  CONSTRAINT pk_dim_store PRIMARY KEY (store_key)
)
COMMENT 'Gold: store dimension, SCD Type 1. No unknown member: orders without a valid store are rejected in silver.';

MERGE INTO gold_efkleia_urbanmart.dim_store AS t
USING (
  SELECT
    store_id, store_name, store_type,
    CASE WHEN store_type = 'Online' THEN 'Online' ELSE 'Physical' END AS channel,
    city, region, open_date, size_sqm,
    sha2(concat_ws('||', coalesce(store_name, ''), coalesce(store_type, ''), coalesce(city, ''),
                         coalesce(region, ''), coalesce(CAST(open_date AS STRING), ''),
                         coalesce(CAST(size_sqm AS STRING), '')), 256) AS attr_hash
  FROM silver_efkleia_urbanmart.stores
) AS s
ON t.store_id = s.store_id
WHEN MATCHED AND t._attr_hash <> s.attr_hash THEN UPDATE SET
  t.store_name  = s.store_name,
  t.store_type  = s.store_type,
  t.channel     = s.channel,
  t.city        = s.city,
  t.region      = s.region,
  t.open_date   = s.open_date,
  t.size_sqm    = s.size_sqm,
  t._attr_hash  = s.attr_hash,
  t._updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT
  (store_id, store_name, store_type, channel, city, region, open_date, size_sqm, _attr_hash, _inserted_at)
VALUES
  (s.store_id, s.store_name, s.store_type, s.channel, s.city, s.region, s.open_date, s.size_sqm,
   s.attr_hash, current_timestamp());

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
SELECT 'dim_date' AS dim, COUNT(*) AS total_rows, MIN(date_key) AS min_key, MAX(date_key) AS max_key
FROM gold_efkleia_urbanmart.dim_date                       -- expect 731 (2024 is a leap year)
UNION ALL
SELECT 'dim_customer', COUNT(*), MIN(customer_key), MAX(customer_key)
FROM gold_efkleia_urbanmart.dim_customer                   -- expect 501 (500 + unknown), min -1
UNION ALL
SELECT 'dim_product', COUNT(*), MIN(product_key), MAX(product_key)
FROM gold_efkleia_urbanmart.dim_product                    -- expect 76 (75 + unknown, P9999 excluded), min -1
UNION ALL
SELECT 'dim_store', COUNT(*), MIN(store_key), MAX(store_key)
FROM gold_efkleia_urbanmart.dim_store;                     -- expect 13

dim,total_rows,min_key,max_key
dim_date,731,20240101,20251231
dim_customer,501,-1,500
dim_product,76,-1,75
dim_store,13,1,13


In [0]:
CREATE TABLE IF NOT EXISTS gold_efkleia_urbanmart.fact_sales (
  order_line_id      STRING        NOT NULL COMMENT 'Degenerate dimension; one row per order line (grain).',
  order_id           STRING        NOT NULL COMMENT 'Degenerate dimension; source order number.',
  date_key           INT           NOT NULL COMMENT 'FK to dim_date (order date).',
  customer_key       BIGINT        NOT NULL COMMENT 'FK to dim_customer; -1 = guest / unknown.',
  customer_scd2_key  BIGINT                 COMMENT 'FK to dim_customer_scd2: customer version valid on the order date; -1 = guest.',  -- SCD2
  product_key        BIGINT        NOT NULL COMMENT 'FK to dim_product; -1 = unknown product.',
  store_key          BIGINT        NOT NULL COMMENT 'FK to dim_store.',
  order_status       STRING        NOT NULL COMMENT 'Completed / Returned / Cancelled.',
  payment_method     STRING                 COMMENT 'Card / Cash / Mobile Wallet.',
  quantity           INT           NOT NULL COMMENT 'Units sold (additive).',
  unit_price         DECIMAL(10,2) NOT NULL COMMENT 'Selling price per unit (non-additive).',
  discount_pct       DECIMAL(5,4)  NOT NULL COMMENT 'Discount fraction (non-additive).',
  gross_amount       DECIMAL(12,2) NOT NULL COMMENT 'quantity x unit_price (additive).',
  discount_amount    DECIMAL(12,2) NOT NULL COMMENT 'gross x discount_pct (additive).',
  net_amount         DECIMAL(12,2) NOT NULL COMMENT 'gross - discount (additive).',
  cost_amount        DECIMAL(12,2) NOT NULL COMMENT 'quantity x product unit_cost; 0 for unknown product (additive).',
  margin_amount      DECIMAL(12,2) NOT NULL COMMENT 'net - cost (additive).',
  _source_file       STRING        NOT NULL COMMENT 'Lineage: source file of the order line.',
  _gold_loaded_at    TIMESTAMP     NOT NULL COMMENT 'Gold load timestamp.',
  CONSTRAINT pk_fact_sales  PRIMARY KEY (order_line_id),
  CONSTRAINT fk_fact_date          FOREIGN KEY (date_key)          REFERENCES gold_efkleia_urbanmart.dim_date (date_key),
  CONSTRAINT fk_fact_customer      FOREIGN KEY (customer_key)      REFERENCES gold_efkleia_urbanmart.dim_customer (customer_key),
  CONSTRAINT fk_fact_customer_scd2 FOREIGN KEY (customer_scd2_key) REFERENCES gold_efkleia_urbanmart.dim_customer_scd2 (customer_scd2_key),  -- SCD2
  CONSTRAINT fk_fact_product       FOREIGN KEY (product_key)       REFERENCES gold_efkleia_urbanmart.dim_product (product_key),
  CONSTRAINT fk_fact_store         FOREIGN KEY (store_key)         REFERENCES gold_efkleia_urbanmart.dim_store (store_key)
)
CLUSTER BY (date_key)
COMMENT 'Gold: sales fact, grain = one order line. Includes Completed, Returned and Cancelled orders.';

In [0]:
-- ONE-TIME MIGRATION (already applied):adds the SCD2 key to the existing fact_sales table.

-- ALTER TABLE gold_efkleia_urbanmart.fact_sales
--   ADD COLUMNS (customer_scd2_key BIGINT COMMENT 'FK to dim_customer_scd2: customer version valid on the order date; -1 = guest.' AFTER customer_key);

-- ALTER TABLE gold_efkleia_urbanmart.fact_sales
--   ADD CONSTRAINT fk_fact_customer_scd2 FOREIGN KEY (customer_scd2_key)
--   REFERENCES gold_efkleia_urbanmart.dim_customer_scd2 (customer_scd2_key);

In [0]:
DESCRIBE TABLE gold_efkleia_urbanmart.fact_sales;

col_name,data_type,comment
order_line_id,string,Degenerate dimension; one row per order line (grain).
order_id,string,Degenerate dimension; source order number.
date_key,int,FK to dim_date (order date).
customer_key,bigint,FK to dim_customer; -1 = guest / unknown.
customer_scd2_key,bigint,FK to dim_customer_scd2: customer version valid on the order date; -1 = guest.
product_key,bigint,FK to dim_product; -1 = unknown product.
store_key,bigint,FK to dim_store.
order_status,string,Completed / Returned / Cancelled.
payment_method,string,Card / Cash / Mobile Wallet.
quantity,int,Units sold (additive).


In [0]:
ALTER TABLE gold_efkleia_urbanmart.fact_sales DROP CONSTRAINT IF EXISTS ck_fact_quantity_positive;
ALTER TABLE gold_efkleia_urbanmart.fact_sales ADD  CONSTRAINT ck_fact_quantity_positive CHECK (quantity > 0);

ALTER TABLE gold_efkleia_urbanmart.fact_sales DROP CONSTRAINT IF EXISTS ck_fact_discount_range;
ALTER TABLE gold_efkleia_urbanmart.fact_sales ADD  CONSTRAINT ck_fact_discount_range CHECK (discount_pct BETWEEN 0 AND 1);

ALTER TABLE gold_efkleia_urbanmart.fact_sales DROP CONSTRAINT IF EXISTS ck_fact_net_not_negative;
ALTER TABLE gold_efkleia_urbanmart.fact_sales ADD  CONSTRAINT ck_fact_net_not_negative CHECK (net_amount >= 0);

In [0]:
INSERT OVERWRITE gold_efkleia_urbanmart.fact_sales
  (order_line_id, order_id, date_key, customer_key, customer_scd2_key, product_key, store_key,   -- SCD2: customer_scd2_key
   order_status, payment_method, quantity, unit_price, discount_pct,
   gross_amount, discount_amount, net_amount, cost_amount, margin_amount,
   _source_file, _gold_loaded_at)
SELECT
  i.order_line_id,
  i.order_id,
  d.date_key,
  coalesce(c.customer_key, -1)                                          AS customer_key,
  coalesce(h.customer_scd2_key, -1)                                     AS customer_scd2_key,   -- SCD2
  coalesce(p.product_key,  -1)                                          AS product_key,
  s.store_key,
  o.order_status,
  o.payment_method,
  i.quantity,
  i.unit_price,
  i.discount_pct,
  i.gross_amount,
  i.discount_amount,
  i.net_amount,
  CAST(i.quantity * coalesce(p.unit_cost, 0) AS DECIMAL(12,2))          AS cost_amount,
  CAST(i.net_amount - i.quantity * coalesce(p.unit_cost, 0) AS DECIMAL(12,2)) AS margin_amount,
  i._source_file,
  current_timestamp()
FROM silver_efkleia_urbanmart.order_items i
JOIN silver_efkleia_urbanmart.orders o
  ON i.order_id = o.order_id
LEFT JOIN gold_efkleia_urbanmart.dim_date d
  ON d.full_date = o.order_date
LEFT JOIN gold_efkleia_urbanmart.dim_customer c
  ON c.customer_id = o.customer_id
LEFT JOIN gold_efkleia_urbanmart.dim_customer_scd2 h                    -- SCD2: point-in-time join
  ON  h.customer_id = o.customer_id
  AND o.order_date BETWEEN h.valid_from AND h.valid_to
LEFT JOIN gold_efkleia_urbanmart.dim_product p
  ON p.product_id = i.product_id
LEFT JOIN gold_efkleia_urbanmart.dim_store s
  ON s.store_id = o.store_id;

num_affected_rows,num_inserted_rows
23997,23997


In [0]:
SELECT
  COUNT(*)                          AS fact_rows,        -- expect 23997
  COUNT(DISTINCT order_line_id)     AS distinct_lines,   -- expect 23997
  COUNT_IF(customer_key = -1)       AS guest_lines,      -- expect 989
  COUNT_IF(product_key  = -1)       AS unknown_product,  -- expect 1
  SUM(net_amount)                   AS total_net,        -- expect 10063191.03
  SUM(cost_amount)                  AS total_cost,       -- expect 7099875.18
  SUM(margin_amount)                AS total_margin      -- expect 2963315.85
FROM gold_efkleia_urbanmart.fact_sales;

fact_rows,distinct_lines,guest_lines,unknown_product,total_net,total_cost,total_margin
23997,23997,989,1,10063191.03,7099875.18,2963315.85


**Result:** dim_date 731 · dim_customer 501 · dim_product 76 · dim_store 13 · ❗ dim_customer_scd2 525.
fact_sales 23,997 rows (Completed 21,739 · Returned 1,380 · Cancelled 878), 989 guest lines (customer −1), 1 unknown-product line (product −1).
Net amount 10,063,191.03 · cost 7,099,875.18 · margin 2,963,315.85.
Re-running the dimension MERGEs affects 0 rows (idempotent).
❗ SCD2: 310 fact lines from 24 customers point to the version valid on the order date (e.g. C0027 switches from Retail/Patras to Loyalty Gold/Heraklion on 2025-04-12).

In [0]:
SELECT order_status, COUNT(*) AS lines
FROM gold_efkleia_urbanmart.fact_sales
GROUP BY order_status
ORDER BY lines DESC;
-- expect: Completed 21739 · Returned 1380 · Cancelled 878

order_status,lines
Completed,21739
Returned,1380
Cancelled,878


In [0]:
SELECT
  COUNT(*)                              AS fact_rows,          -- expect 23997 (unchanged)
  COUNT_IF(f.customer_scd2_key = -1)    AS guest_lines,        -- expect 989
  COUNT_IF(h.version_num = 2)           AS lines_on_version_2, -- expect 310
  SUM(f.net_amount)                     AS total_net           -- expect 10063191.03 (unchanged)
FROM gold_efkleia_urbanmart.fact_sales f
JOIN gold_efkleia_urbanmart.dim_customer_scd2 h ON f.customer_scd2_key = h.customer_scd2_key;

fact_rows,guest_lines,lines_on_version_2,total_net
23997,989,310,10063191.03


In [0]:
SELECT d.full_date, f.order_id, h.version_num, h.city, h.segment
FROM gold_efkleia_urbanmart.fact_sales f
JOIN gold_efkleia_urbanmart.dim_date d          ON f.date_key = d.date_key
JOIN gold_efkleia_urbanmart.dim_customer_scd2 h ON f.customer_scd2_key = h.customer_scd2_key
WHERE h.customer_id = 'C0027'
ORDER BY d.full_date;

full_date,order_id,version_num,city,segment
2024-05-13,O000531,1,Patras,Retail
2024-05-31,O005749,1,Patras,Retail
2024-05-31,O005749,1,Patras,Retail
2024-05-31,O005749,1,Patras,Retail
2024-09-01,O005742,1,Patras,Retail
2024-09-01,O005742,1,Patras,Retail
2024-09-01,O005742,1,Patras,Retail
2024-09-17,O001108,1,Patras,Retail
2024-09-17,O001108,1,Patras,Retail
2024-09-17,O001108,1,Patras,Retail


### ERD – Star schema
*Reminder: the diagram shows ey26-023 | Chrysanthi Chamou due to the  account mix-up. Author: Efkleia Karagkiozi.*
ERD: see `erd_star_schema.png` in the zip and in the documentation.
**Generated with Catalog Explorer from the PK/FK constraints created in Phase 3;
it confirms that the implemented model matches the design above.*